# 02 - Aplicación en Clasificación de Texto y Benchmark Comparativo

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno estudiaremos la aplicación más icónica y exitosa de Naïve Bayes: el **procesamiento de lenguaje natural (NLP)** y la clasificación de documentos mediante `MultinomialNB` y matrices de frecuencias TF-IDF. Además, concluiremos con un **benchmark comparativo integral** midiendo la precisión, velocidad de entrenamiento y tiempo de inferencia de los 5 algoritmos de clasificación del curso.

---

### Contenidos
1. Vectorización de Texto: Bag of Words (`CountVectorizer`) y TF-IDF (`TfidfVectorizer`)
2. Clasificación de Texto con `MultinomialNB`
3. Evaluación de Métricas Críticas: Precisión, Recall y F1-Score en Detección de Mensajes
4. Fronteras de Decisión en 2D para Naïve Bayes
5. Benchmark Integral de Clasificadores: Regresión Logística, KNN, Árboles, Random Forest y Naïve Bayes
6. Conclusiones Generales del Módulo de Clasificación


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.inspection import DecisionBoundaryDisplay

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Clasificación de Texto con Multinomial Naïve Bayes

Creamos un corpus educativo de mensajes para clasificación binaria de spam / no spam (ham):


In [ ]:
# Dataset sintetico representativo de mensajes de texto
textos = [
    "Felicidades ha ganado un premio en efectivo llame ahora para cobrar",
    "Urgente su cuenta bancaria ha sido bloqueada haga clic en el enlace",
    "Oferta exclusiva gane dinero rapido desde casa sin inversion",
    "Reclame su boleto de loteria gratis hoy premio garantizado",
    "Promocion de prestamo preaprobado tasa cero solicite ya",
    "Hola como estas nos vemos en la cafeteria para almorzar",
    "La reunion del equipo de proyecto se reprogramo para el viernes a las 10",
    "Te envio el informe de analisis financiero adjunto en formato pdf",
    "Recuerda comprar leche y pan cuando regreses del trabajo",
    "Confirmada tu asistencia para la sesion de machine learning de manana",
    "Premio especial de vacaciones viaje gratis a cancun llame ya",
    "Podrias revisar el documento de especificaciones tecnicas por favor"
]

etiquetas = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0] # 1: Spam, 0: Legitimo (Ham)

# Pipeline de vectorizacion TF-IDF y clasificador MultinomialNB
pipeline_spam = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True)),
    ('nb', MultinomialNB(alpha=1.0)) # Laplace smoothing
])

pipeline_spam.fit(textos, etiquetas)

# Pruebas con nuevos mensajes no vistos
nuevos_mensajes = [
    "Reclame su premio exclusivo en efectivo llamando hoy",
    "Nos vemos manana en la oficina para revisar el informe",
    "Gane un viaje gratis haciendo clic en este enlace"
]

predicciones = pipeline_spam.predict(nuevos_mensajes)
probabilidades = pipeline_spam.predict_proba(nuevos_mensajes)

for msg, pred, prob in zip(nuevos_mensajes, predicciones, probabilidades):
    clase_str = "SPAM" if pred == 1 else "LEGITIMO"
    print(f"Mensaje: '{msg}'")
    print(f"-> Prediccion: {clase_str} (Prob Spam: {prob[1]*100:.1f}% | Prob Legitimo: {prob[0]*100:.1f}%)
")


## 2. Fronteras de Decisión en 2D: Naïve Bayes

Visualizamos cómo GaussianNB construye fronteras parabólicas o elípticas en un espacio bidimensional debido a la combinación de varianzas gaussianas independientes:


In [ ]:
X_2d, y_2d = make_moons(n_samples=250, noise=0.25, random_state=42)

gnb_2d = GaussianNB().fit(X_2d, y_2d)

plt.figure(figsize=(8, 5))
DecisionBoundaryDisplay.from_estimator(gnb_2d, X_2d, cmap='coolwarm', alpha=0.6, response_method='predict')
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap='coolwarm', edgecolor='k', s=45)
plt.title(f'Frontera de Decision de GaussianNB (Acc: {gnb_2d.score(X_2d, y_2d)*100:.1f}%)', fontweight='bold')
plt.xlabel('Caracteristica 1')
plt.ylabel('Caracteristica 2')
plt.tight_layout()
plt.show()


## 3. Benchmark Integral de Clasificadores de Machine Learning

Comparamos el rendimiento de los 5 métodos principales de clasificación abordados en este curso utilizando el dataset `load_breast_cancer`:
1. Regresión Logística (L2)
2. K-Nearest Neighbors (KNN, $K=7$)
3. Árbol de Decisión (CART, $max\_depth=4$)
4. Random Forest ($n=100$)
5. Gaussian Naïve Bayes


In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'KNN (K=7)': KNeighborsClassifier(n_neighbors=7),
    'Arbol de Decision': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gaussian Naive Bayes': GaussianNB()
}

resultados = []

for nombre, clf in modelos.items():
    # Medicion tiempo de entrenamiento
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    t_train = (time.perf_counter() - t0) * 1000 # milisegundos
    
    # Medicion tiempo de inferencia en test
    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    t_pred = (time.perf_counter() - t0) * 1000 # milisegundos
    
    acc = accuracy_score(y_test, y_pred)
    
    resultados.append({
        'Modelo': nombre,
        'Exactitud Test (%)': acc * 100,
        'Tiempo Entrenamiento (ms)': t_train,
        'Tiempo Inferencia (ms)': t_pred
    })

df_bench = pd.DataFrame(resultados).sort_values(by='Exactitud Test (%)', ascending=False)
print("Resultados del Benchmark de Clasificacion:
")
print(df_bench.to_string(index=False))


In [ ]:
# Visualizacion comparativa del Benchmark
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_bench.plot(kind='barh', x='Modelo', y='Exactitud Test (%)', ax=axes[0], color='#2b5c8f', legend=False)
axes[0].set_title('Exactitud en Conjunto de Prueba', fontweight='bold')
axes[0].set_xlim(85, 100)
axes[0].set_xlabel('Exactitud (%)')
axes[0].invert_yaxis()

df_bench.plot(kind='barh', x='Modelo', y='Tiempo Entrenamiento (ms)', ax=axes[1], color='#d95f02', legend=False)
axes[1].set_title('Tiempo de Entrenamiento (Milisegundos)', fontweight='bold')
axes[1].set_xlabel('Milisegundos (Escala Lineal)')
axes[1].invert_yaxis()

plt.suptitle('Comparativa Global de Algoritmos de Clasificacion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Conclusiones y Resumen del Módulo

- **Naïve Bayes**: Sobresale por su velocidad imbatible y simplicidad en procesamiento de texto, datos dispersos o sistemas en tiempo real.
- **Regresión Logística**: Excelente baseline probabilístico, transparente e interpretable en datos linealmente separables.
- **KNN**: Efectivo en baja dimensionalidad y geometrías arbitrarias, pero costoso computacionalmente en inferencia y sensible a escalas.
- **Árboles de Decisión**: Altamente visuales e interpretables, pero susceptibles al sobreajuste si no se podan.
- **Random Forest**: El estándar de oro en datos tabulares con alta exactitud, robustez ante ruido e insensibilidad al sobreajuste por número de estimadores.
